# 🌲 Modèle 1 : Isolation Forest — Détection d'Anomalies dans les Logs d'Authentification

## Objectif
Utiliser l'algorithme **Isolation Forest** pour détecter automatiquement les comportements suspects
et les connexions inhabituelles dans les logs d'authentification du dataset **LANL Comprehensive Cyber-Security Events**.

### Principe de l'Isolation Forest
L'Isolation Forest repose sur l'idée que les **anomalies sont plus faciles à isoler** que les points normaux :
- Il construit un ensemble d'arbres de décision aléatoires (forêt).
- Chaque arbre partitionne les données en choisissant aléatoirement une feature et un seuil.
- Les **anomalies** nécessitent **moins de partitions** (profondeur plus faible) pour être isolées.
- Un **score d'anomalie** est calculé à partir de la profondeur moyenne dans la forêt.

### Plan de ce notebook
1. Imports et configuration
2. Chargement des données (LANL ou simulation)
3. Prétraitement et Feature Engineering
4. Entraînement du modèle Isolation Forest
5. Analyse des scores d'anomalie
6. Évaluation avec les labels Red Team
7. Visualisations
8. Conclusion

---

**Auteurs** : Ouhmmou Samir & Mezine Athman  
**Dataset** : LANL Comprehensive Cyber-Security Events

## 1. 📦 Imports et Configuration

In [1]:
# === Bibliothèques principales ===
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    precision_recall_curve, auc, 
    roc_auc_score, f1_score,
    precision_score, recall_score
)

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Configuration
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

print("✅ Toutes les bibliothèques sont chargées avec succès.")
print(f"   Pandas  : {pd.__version__}")
print(f"   NumPy   : {np.__version__}")

✅ Toutes les bibliothèques sont chargées avec succès.
   Pandas  : 3.1.1
   NumPy   : 2.5.4


## 2. 📥 Chargement des Données

### Format du dataset LANL `auth.txt.gz`
Le fichier d'authentification LANL contient les colonnes suivantes :

| Colonne | Description |
|---------|-------------|
| `time` | Timestamp (en secondes depuis le début) |
| `src_user` | Utilisateur source (ex: `U1`, `U2`, ...) |
| `dst_user` | Utilisateur destination |
| `src_computer` | Machine source (ex: `C1`, `C2`, ...) |
| `dst_computer` | Machine destination |
| `auth_type` | Type d'authentification (ex: `Kerberos`, `NTLM`, ...) |
| `logon_type` | Type de connexion (ex: `Network`, `Interactive`, ...) |
| `auth_orient` | Orientation (ex: `LogOn`, `LogOff`, ...) |
| `result` | Résultat (`Success` ou `Fail`) |

> ⚠️ Si le fichier LANL n'est pas disponible localement, nous générons un **échantillon synthétique réaliste** qui reproduit fidèlement la structure et les patterns du dataset LANL.

In [2]:
import os

# ======================================================================
# OPTION 1 : Charger le vrai dataset LANL
# ======================================================================
# Modifiez ce chemin si vous avez téléchargé les fichiers LANL
AUTH_PATH = '../dataset/auth.txt.gz'
REDTEAM_PATH = '../dataset/redteam.txt.gz'

LANL_COLUMNS = ['time', 'src_user', 'dst_user', 'src_computer', 
                'dst_computer', 'auth_type', 'logon_type', 
                'auth_orient', 'result']

REDTEAM_COLUMNS = ['time', 'src_user', 'src_computer', 'dst_computer']

USE_REAL_DATA = os.path.exists(AUTH_PATH)

if USE_REAL_DATA:
    print("📂 Chargement du dataset LANL réel...")
    # Charger un échantillon de 2 millions de lignes pour la mémoire
    N_ROWS = 2_000_000
    auth_df = pd.read_csv(
        AUTH_PATH, sep=',', header=None,
        names=LANL_COLUMNS, nrows=N_ROWS
    )
    
    # Charger les événements Red Team
    if os.path.exists(REDTEAM_PATH):
        redteam_df = pd.read_csv(
            REDTEAM_PATH, sep=',', header=None,
            names=REDTEAM_COLUMNS
        )
        print(f"   🔴 Red Team events : {len(redteam_df):,}")
    else:
        redteam_df = pd.DataFrame(columns=REDTEAM_COLUMNS)
        print("   ⚠️ Fichier redteam.txt.gz non trouvé.")
    
    print(f"   ✅ {len(auth_df):,} événements d'authentification chargés.")

else:
    print("⚠️ Dataset LANL non trouvé. Génération de données synthétiques...")
    print("   (Pour utiliser le vrai dataset, placez auth.txt.gz et")
    print("    redteam.txt.gz dans le dossier ../dataset/)")
    print()

⚠️ Dataset LANL non trouvé. Génération de données synthétiques...
   (Pour utiliser le vrai dataset, placez auth.txt.gz et
    redteam.txt.gz dans le dossier ../dataset/)



In [3]:
# ======================================================================
# OPTION 2 : Génération de données synthétiques réalistes
# ======================================================================
# Ce bloc s'exécute uniquement si le dataset LANL n'est pas disponible.
# Les données simulées reproduisent les patterns typiques du LANL :
#   - Utilisateurs normaux avec comportement régulier
#   - Utilisateurs Red Team avec comportement anormal
#   - Pics d'activité, connexions hors heures, etc.

if not USE_REAL_DATA:
    np.random.seed(42)
    
    N_EVENTS = 500_000       # Nombre total d'événements
    N_USERS = 200            # Nombre d'utilisateurs normaux
    N_COMPUTERS = 500        # Nombre de machines
    N_RED_TEAM_USERS = 5     # Nombre d'utilisateurs compromis
    DURATION_SECONDS = 7 * 86400  # 7 jours de données
    
    # --- Listes d'entités ---
    users = [f'U{i}' for i in range(1, N_USERS + 1)]
    computers = [f'C{i}' for i in range(1, N_COMPUTERS + 1)]
    auth_types = ['Kerberos', 'NTLM', 'Negotiate', 'MICROSOFT_AUTHENTICATION_PACKAGE_V1_0']
    logon_types = ['Network', 'Interactive', 'Batch', 'Service', 'RemoteInteractive']
    auth_orients = ['LogOn', 'LogOff', 'TGS', 'TGT', 'AuthMap']
    
    # --- Utilisateurs Red Team (compromis) ---
    red_team_users = np.random.choice(users, N_RED_TEAM_USERS, replace=False)
    print(f"   🔴 Utilisateurs Red Team simulés : {list(red_team_users)}")
    
    # --- Générer les événements normaux ---
    # Les utilisateurs normaux ont des patterns réguliers :
    #   - Heures de travail (8h-18h)
    #   - 2-5 machines habituelles
    #   - Principalement Kerberos
    #   - Taux d'échec < 2%
    
    events = []
    
    # Assigner à chaque utilisateur ses machines habituelles
    user_machines = {}
    for u in users:
        n_machines = np.random.randint(2, 6)
        user_machines[u] = np.random.choice(computers, n_machines, replace=False)
    
    # Événements normaux (95% des données)
    n_normal = int(N_EVENTS * 0.95)
    for _ in range(n_normal):
        user = np.random.choice(users)
        # Temps : distribution biaisée vers les heures de travail
        day = np.random.randint(0, 7)
        # 85% pendant les heures de travail (8h-18h)
        if np.random.random() < 0.85:
            hour = np.random.randint(8, 18)
        else:
            hour = np.random.choice([0,1,2,3,4,5,6,7,18,19,20,21,22,23])
        minute = np.random.randint(0, 60)
        second = np.random.randint(0, 60)
        time_val = day * 86400 + hour * 3600 + minute * 60 + second
        
        # Machine depuis les machines habituelles (90%) ou nouvelle (10%)
        if np.random.random() < 0.9:
            dst = np.random.choice(user_machines[user])
        else:
            dst = np.random.choice(computers)
        src = np.random.choice(user_machines[user])
        
        events.append({
            'time': time_val,
            'src_user': user,
            'dst_user': user,
            'src_computer': src,
            'dst_computer': dst,
            'auth_type': np.random.choice(auth_types, p=[0.6, 0.25, 0.1, 0.05]),
            'logon_type': np.random.choice(logon_types, p=[0.5, 0.25, 0.1, 0.1, 0.05]),
            'auth_orient': np.random.choice(auth_orients, p=[0.4, 0.3, 0.15, 0.1, 0.05]),
            'result': np.random.choice(['Success', 'Fail'], p=[0.98, 0.02]),
            'is_red_team': 0
        })
    
    # Événements Red Team (5% des données) — comportements anormaux
    # Caractéristiques : 
    #   - Connexions hors heures (nuit, week-end)
    #   - Accès à de nombreuses machines inhabituelles
    #   - Taux d'échec élevé (brute force)
    #   - Types d'authentification variés
    n_redteam = N_EVENTS - n_normal
    for _ in range(n_redteam):
        user = np.random.choice(red_team_users)
        day = np.random.randint(0, 7)
        # 70% hors heures de travail
        if np.random.random() < 0.70:
            hour = np.random.choice([0,1,2,3,4,5,6,22,23])
        else:
            hour = np.random.randint(8, 18)
        minute = np.random.randint(0, 60)
        second = np.random.randint(0, 60)
        time_val = day * 86400 + hour * 3600 + minute * 60 + second
        
        # Accès à des machines aléatoires (inhabituelles)
        dst = np.random.choice(computers)
        src = np.random.choice(computers)
        
        events.append({
            'time': time_val,
            'src_user': user,
            'dst_user': np.random.choice(users),  # Usurpation possible
            'src_computer': src,
            'dst_computer': dst,
            'auth_type': np.random.choice(auth_types, p=[0.2, 0.4, 0.2, 0.2]),
            'logon_type': np.random.choice(logon_types, p=[0.3, 0.2, 0.15, 0.15, 0.2]),
            'auth_orient': np.random.choice(auth_orients, p=[0.5, 0.1, 0.15, 0.15, 0.1]),
            'result': np.random.choice(['Success', 'Fail'], p=[0.65, 0.35]),
            'is_red_team': 1
        })
    
    # Créer le DataFrame
    auth_df = pd.DataFrame(events)
    auth_df = auth_df.sort_values('time').reset_index(drop=True)
    
    # Créer le DataFrame Red Team
    redteam_df = auth_df[auth_df['is_red_team'] == 1][['time', 'src_user', 'src_computer', 'dst_computer']].copy()
    
    print(f"\n   ✅ {len(auth_df):,} événements synthétiques générés.")
    print(f"   🔴 {len(redteam_df):,} événements Red Team")
    print(f"   🟢 {len(auth_df) - len(redteam_df):,} événements normaux")
    print(f"   📅 Période : {auth_df['time'].min()} → {auth_df['time'].max()} secondes (≈ 7 jours)")

   🔴 Utilisateurs Red Team simulés : [np.str_('U96'), np.str_('U16'), np.str_('U31'), np.str_('U159'), np.str_('U129')]

   ✅ 500,000 événements synthétiques générés.
   🔴 25,000 événements Red Team
   🟢 475,000 événements normaux
   📅 Période : 0 → 604799 secondes (≈ 7 jours)


In [ ]:
# === Aperçu des données ===
print("=" * 70)
print("📊 APERÇU DU DATASET D'AUTHENTIFICATION")
print("=" * 70)
print(f"\nShape : {auth_df.shape}")
print(f"Colonnes : {list(auth_df.columns)}")
print()
auth_df.head(10)

In [ ]:
# === Statistiques descriptives ===
print("\n📈 Statistiques descriptives :")
print(f"  Utilisateurs uniques (src_user) : {auth_df['src_user'].nunique()}")
print(f"  Machines source uniques         : {auth_df['src_computer'].nunique()}")
print(f"  Machines destination uniques     : {auth_df['dst_computer'].nunique()}")
print(f"  Types d'authentification         : {auth_df['auth_type'].nunique()}")
print(f"  Types de connexion               : {auth_df['logon_type'].nunique()}")
print()
print("Résultats de connexion :")
print(auth_df['result'].value_counts())
print()
print("Types d'authentification :")
print(auth_df['auth_type'].value_counts())

## 3. 🧹 Prétraitement et Feature Engineering

### Stratégie
On agrège les événements par **utilisateur** et par **fenêtre temporelle d'une heure** pour construire des features comportementales.

### Features extraites

| Feature | Description | Intuition |
|---------|-------------|----------|
| `login_count` | Nombre total de connexions | Volume d'activité |
| `unique_src_machines` | Nombre de machines source uniques | Mobilité |
| `unique_dst_machines` | Nombre de machines destination uniques | Étendue d'accès |
| `failed_count` | Nombre de connexions échouées | Tentatives suspectes |
| `failure_ratio` | Ratio échecs / total | Brute force |
| `auth_types` | Nombre de types d'authentification | Diversité des méthodes |
| `logon_types` | Nombre de types de connexion | Diversité des modes |
| `off_hours_ratio` | Ratio connexions hors heures (20h-7h) | Activité nocturne |
| `unique_dst_users` | Nombre d'utilisateurs destination | Usurpation / lateral movement |

In [ ]:
# === 3.1 Ajout des colonnes temporelles ===

# Convertir le timestamp en heure et jour
auth_df['hour'] = (auth_df['time'] % 86400) // 3600       # Heure du jour (0-23)
auth_df['day'] = auth_df['time'] // 86400                  # Jour (0, 1, 2, ...)
auth_df['time_window'] = auth_df['day'].astype(str) + '_' + auth_df['hour'].astype(str).str.zfill(2)

# Indicateur heures creuses (20h-7h)
auth_df['is_off_hours'] = ((auth_df['hour'] >= 20) | (auth_df['hour'] < 7)).astype(int)

# Indicateur échec
auth_df['is_failure'] = (auth_df['result'] == 'Fail').astype(int)

print("✅ Colonnes temporelles ajoutées.")
print(f"   Jours couverts : {auth_df['day'].nunique()}")
print(f"   Fenêtres temporelles : {auth_df['time_window'].nunique()}")
auth_df[['time', 'day', 'hour', 'time_window', 'is_off_hours', 'is_failure']].head(10)

In [ ]:
# === 3.2 Agrégation : Features par utilisateur et par fenêtre temporelle ===

features_df = auth_df.groupby(['src_user', 'time_window']).agg(
    login_count       = ('time', 'count'),
    unique_src_machines = ('src_computer', 'nunique'),
    unique_dst_machines = ('dst_computer', 'nunique'),
    failed_count      = ('is_failure', 'sum'),
    auth_types        = ('auth_type', 'nunique'),
    logon_types       = ('logon_type', 'nunique'),
    off_hours_count   = ('is_off_hours', 'sum'),
    unique_dst_users  = ('dst_user', 'nunique'),
    day               = ('day', 'first'),
    hour              = ('hour', 'first'),
).reset_index()

# Calculer les ratios
features_df['failure_ratio'] = features_df['failed_count'] / features_df['login_count']
features_df['off_hours_ratio'] = features_df['off_hours_count'] / features_df['login_count']

# Remplacer les NaN
features_df = features_df.fillna(0)

print(f"✅ Feature Engineering terminé.")
print(f"   Nombre d'observations (user × time_window) : {len(features_df):,}")
print(f"   Nombre de features : {len(features_df.columns) - 4}")
print()
features_df.head(10)

In [ ]:
# === 3.3 Labellisation avec les événements Red Team ===

# Identifier les utilisateurs et fenêtres temporelles Red Team
if not USE_REAL_DATA:
    # En mode synthétique, le label est déjà dans auth_df
    rt_labels = auth_df.groupby(['src_user', 'time_window'])['is_red_team'].max().reset_index()
    features_df = features_df.merge(rt_labels, on=['src_user', 'time_window'], how='left')
    features_df['is_red_team'] = features_df['is_red_team'].fillna(0).astype(int)
else:
    # En mode LANL réel, croiser avec redteam_df
    if len(redteam_df) > 0:
        redteam_df['hour'] = (redteam_df['time'] % 86400) // 3600
        redteam_df['day'] = redteam_df['time'] // 86400
        redteam_df['time_window'] = redteam_df['day'].astype(str) + '_' + redteam_df['hour'].astype(str).str.zfill(2)
        
        rt_set = set(zip(redteam_df['src_user'], redteam_df['time_window']))
        features_df['is_red_team'] = features_df.apply(
            lambda row: 1 if (row['src_user'], row['time_window']) in rt_set else 0,
            axis=1
        )
    else:
        features_df['is_red_team'] = 0

print(f"\n📊 Distribution des labels :")
print(f"   Normal (0)    : {(features_df['is_red_team'] == 0).sum():,}")
print(f"   Red Team (1)  : {(features_df['is_red_team'] == 1).sum():,}")
print(f"   Ratio Red Team : {features_df['is_red_team'].mean():.4%}")

In [ ]:
# === 3.4 Statistiques des features ===

FEATURE_COLS = ['login_count', 'unique_src_machines', 'unique_dst_machines',
                'failed_count', 'failure_ratio', 'auth_types', 'logon_types',
                'off_hours_ratio', 'unique_dst_users']

print("\n📊 Statistiques des features :")
features_df[FEATURE_COLS].describe().round(3)

In [ ]:
# === 3.5 Visualisation de la distribution des features ===

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
fig.suptitle('Distribution des Features par Label (Normal vs Red Team)', 
             fontsize=16, fontweight='bold', y=1.02)

for idx, feat in enumerate(FEATURE_COLS):
    ax = axes[idx // 3, idx % 3]
    
    # Séparer normal et red team
    normal = features_df[features_df['is_red_team'] == 0][feat]
    redteam = features_df[features_df['is_red_team'] == 1][feat]
    
    ax.hist(normal, bins=50, alpha=0.6, label='Normal', color='steelblue', density=True)
    ax.hist(redteam, bins=50, alpha=0.6, label='Red Team', color='crimson', density=True)
    ax.set_title(feat, fontsize=12, fontweight='bold')
    ax.legend()
    ax.set_ylabel('Densité')

plt.tight_layout()
plt.savefig('../results/figures/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure sauvegardée : ../results/figures/feature_distributions.png")

## 4. 🤖 Entraînement du Modèle Isolation Forest

### Paramètres du modèle

| Paramètre | Valeur | Description |
|-----------|--------|-------------|
| `n_estimators` | 200 | Nombre d'arbres dans la forêt |
| `contamination` | 0.05 | Proportion estimée d'anomalies (5%) |
| `max_samples` | 'auto' | Nombre d'échantillons par arbre |
| `max_features` | 1.0 | Toutes les features utilisées |
| `random_state` | 42 | Reproductibilité |

In [ ]:
# === 4.1 Normalisation des features ===

scaler = StandardScaler()
X = features_df[FEATURE_COLS].copy()
X_scaled = scaler.fit_transform(X)

print("✅ Normalisation avec StandardScaler terminée.")
print(f"   Shape de X_scaled : {X_scaled.shape}")
print(f"   Moyennes après normalisation : {X_scaled.mean(axis=0).round(4)}")
print(f"   Écarts-types après normalisation : {X_scaled.std(axis=0).round(4)}")

In [ ]:
# === 4.2 Entraînement de l'Isolation Forest ===

print("🌲 Entraînement du modèle Isolation Forest...")
print()

iso_forest = IsolationForest(
    n_estimators=200,          # Nombre d'arbres
    contamination=0.05,        # 5% d'anomalies attendues
    max_samples='auto',        # min(256, n_samples)
    max_features=1.0,          # Toutes les features
    bootstrap=False,
    n_jobs=-1,                 # Utiliser tous les cœurs
    random_state=42,
    verbose=0
)

# Entraîner le modèle (non supervisé — pas de labels)
iso_forest.fit(X_scaled)

# Prédictions : 1 = normal, -1 = anomalie
predictions = iso_forest.predict(X_scaled)

# Scores d'anomalie (plus négatif = plus anormal)
anomaly_scores = iso_forest.decision_function(X_scaled)

# Ajouter les résultats au DataFrame
features_df['anomaly_prediction'] = predictions
features_df['anomaly_score'] = anomaly_scores
features_df['is_anomaly'] = (predictions == -1).astype(int)

print("✅ Modèle Isolation Forest entraîné avec succès !")
print(f"\n📊 Résultats des prédictions :")
print(f"   Normal (1)   : {(predictions == 1).sum():,} ({(predictions == 1).mean():.1%})")
print(f"   Anomalie (-1) : {(predictions == -1).sum():,} ({(predictions == -1).mean():.1%})")
print(f"\n📈 Scores d'anomalie :")
print(f"   Min    : {anomaly_scores.min():.4f}")
print(f"   Max    : {anomaly_scores.max():.4f}")
print(f"   Moyenne : {anomaly_scores.mean():.4f}")
print(f"   Médiane : {np.median(anomaly_scores):.4f}")

## 5. 🔍 Analyse des Scores d'Anomalie

In [ ]:
# === 5.1 Distribution des scores d'anomalie ===

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogramme des scores
ax1 = axes[0]
ax1.hist(anomaly_scores[predictions == 1], bins=80, alpha=0.6, 
         label='Normal', color='steelblue', density=True)
ax1.hist(anomaly_scores[predictions == -1], bins=80, alpha=0.6, 
         label='Anomalie', color='crimson', density=True)
ax1.axvline(x=0, color='black', linestyle='--', linewidth=1, label='Seuil (score=0)')
ax1.set_xlabel('Score d\'anomalie', fontsize=12)
ax1.set_ylabel('Densité', fontsize=12)
ax1.set_title('Distribution des Scores d\'Anomalie', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)

# Box plot par label réel
ax2 = axes[1]
data_box = [features_df[features_df['is_red_team'] == 0]['anomaly_score'].values,
            features_df[features_df['is_red_team'] == 1]['anomaly_score'].values]
bp = ax2.boxplot(data_box, labels=['Normal', 'Red Team'], patch_artist=True,
                 boxprops=dict(alpha=0.7))
bp['boxes'][0].set_facecolor('steelblue')
bp['boxes'][1].set_facecolor('crimson')
ax2.set_ylabel('Score d\'anomalie', fontsize=12)
ax2.set_title('Scores d\'Anomalie : Normal vs Red Team', fontsize=14, fontweight='bold')
ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../results/figures/anomaly_scores_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure sauvegardée : ../results/figures/anomaly_scores_distribution.png")

In [ ]:
# === 5.2 Top 20 des observations les plus suspectes ===

top_anomalies = features_df.nsmallest(20, 'anomaly_score')[
    ['src_user', 'time_window', 'login_count', 'unique_dst_machines', 
     'failed_count', 'failure_ratio', 'off_hours_ratio', 
     'anomaly_score', 'is_red_team']
]

print("\n🔴 TOP 20 — Observations les plus suspectes (score le plus bas) :")
print("=" * 90)
top_anomalies

In [ ]:
# === 5.3 Top 20 des utilisateurs les plus suspects (agrégé) ===

user_suspicion = features_df.groupby('src_user').agg(
    mean_anomaly_score = ('anomaly_score', 'mean'),
    min_anomaly_score  = ('anomaly_score', 'min'),
    n_anomalies        = ('is_anomaly', 'sum'),
    total_windows      = ('is_anomaly', 'count'),
    is_red_team        = ('is_red_team', 'max'),
).reset_index()

user_suspicion['anomaly_rate'] = user_suspicion['n_anomalies'] / user_suspicion['total_windows']
user_suspicion = user_suspicion.sort_values('mean_anomaly_score')

top_users = user_suspicion.head(20)

# Visualisation
fig, ax = plt.subplots(figsize=(14, 8))
colors = ['crimson' if rt else 'steelblue' for rt in top_users['is_red_team']]
bars = ax.barh(top_users['src_user'], top_users['mean_anomaly_score'], color=colors, alpha=0.8)
ax.set_xlabel('Score moyen d\'anomalie', fontsize=12)
ax.set_ylabel('Utilisateur', fontsize=12)
ax.set_title('🔴 Top 20 — Utilisateurs les Plus Suspects', fontsize=14, fontweight='bold')
ax.invert_yaxis()

# Légende
legend_handles = [
    mpatches.Patch(color='crimson', alpha=0.8, label='Red Team (confirmé)'),
    mpatches.Patch(color='steelblue', alpha=0.8, label='Normal')
]
ax.legend(handles=legend_handles, loc='lower right', fontsize=11)

plt.tight_layout()
plt.savefig('../results/figures/top_suspect_users.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure sauvegardée : ../results/figures/top_suspect_users.png")

## 6. 📈 Évaluation du Modèle

On compare les anomalies détectées par Isolation Forest avec les labels Red Team réels pour mesurer la performance.

In [ ]:
# === 6.1 Matrice de confusion ===

y_true = features_df['is_red_team'].values
y_pred = features_df['is_anomaly'].values

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Normal (prédit)', 'Anomalie (prédit)'],
            yticklabels=['Normal (réel)', 'Red Team (réel)'],
            ax=ax, cbar_kws={'label': 'Nombre d\'observations'})
ax.set_title('Matrice de Confusion — Isolation Forest', fontsize=14, fontweight='bold')
ax.set_xlabel('Prédiction', fontsize=12)
ax.set_ylabel('Réalité', fontsize=12)

plt.tight_layout()
plt.savefig('../results/figures/confusion_matrix_iforest.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure sauvegardée : ../results/figures/confusion_matrix_iforest.png")

In [ ]:
# === 6.2 Rapport de classification ===

print("\n" + "=" * 60)
print("📊 RAPPORT DE CLASSIFICATION — ISOLATION FOREST")
print("=" * 60)
print()
print(classification_report(y_true, y_pred, 
                            target_names=['Normal', 'Red Team'],
                            digits=4))

# Métriques individuelles
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print(f"\n📌 Métriques clés :")
print(f"   Precision : {precision:.4f}")
print(f"   Recall    : {recall:.4f}")
print(f"   F1-score  : {f1:.4f}")

In [ ]:
# === 6.3 Courbe Precision-Recall ===

# Utiliser les scores d'anomalie (inversés car score plus bas = plus suspect)
y_scores = -anomaly_scores  # Inverser pour que plus haut = plus suspect

precision_curve, recall_curve, thresholds = precision_recall_curve(y_true, y_scores)
pr_auc = auc(recall_curve, precision_curve)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Courbe PR
ax1 = axes[0]
ax1.plot(recall_curve, precision_curve, color='darkorange', linewidth=2, 
         label=f'Isolation Forest (PR-AUC = {pr_auc:.4f})')
ax1.fill_between(recall_curve, precision_curve, alpha=0.2, color='darkorange')
ax1.set_xlabel('Recall', fontsize=12)
ax1.set_ylabel('Precision', fontsize=12)
ax1.set_title('Courbe Precision-Recall', fontsize=14, fontweight='bold')
ax1.legend(loc='best', fontsize=11)
ax1.set_xlim([0, 1])
ax1.set_ylim([0, 1.05])
ax1.grid(True, alpha=0.3)

# F1-score en fonction du seuil
ax2 = axes[1]
f1_scores = 2 * precision_curve[:-1] * recall_curve[:-1] / (
    precision_curve[:-1] + recall_curve[:-1] + 1e-10)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

ax2.plot(thresholds, f1_scores, color='green', linewidth=2)
ax2.axvline(x=best_threshold, color='red', linestyle='--', 
            label=f'Meilleur seuil = {best_threshold:.4f}\nF1 = {best_f1:.4f}')
ax2.set_xlabel('Seuil', fontsize=12)
ax2.set_ylabel('F1-score', fontsize=12)
ax2.set_title('F1-score en Fonction du Seuil', fontsize=14, fontweight='bold')
ax2.legend(loc='best', fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/precision_recall_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Résultats :")
print(f"   PR-AUC : {pr_auc:.4f}")
print(f"   Meilleur seuil : {best_threshold:.4f}")
print(f"   Meilleur F1-score : {best_f1:.4f}")

## 7. 📉 Visualisations Avancées

In [ ]:
# === 7.1 Timeline des anomalies détectées ===

# Regrouper par fenêtre temporelle
timeline = features_df.groupby(['day', 'hour']).agg(
    n_total     = ('is_anomaly', 'count'),
    n_anomalies = ('is_anomaly', 'sum'),
    n_redteam   = ('is_red_team', 'sum'),
    mean_score  = ('anomaly_score', 'mean'),
).reset_index()
timeline['time_index'] = timeline['day'] * 24 + timeline['hour']
timeline = timeline.sort_values('time_index')

fig, axes = plt.subplots(2, 1, figsize=(18, 10), sharex=True)

# Nombre d'anomalies par heure
ax1 = axes[0]
ax1.bar(timeline['time_index'], timeline['n_anomalies'], 
        color='crimson', alpha=0.7, label='Anomalies détectées', width=0.8)
ax1.bar(timeline['time_index'], timeline['n_redteam'], 
        color='darkred', alpha=0.9, label='Red Team réels', width=0.4)
ax1.set_ylabel('Nombre', fontsize=12)
ax1.set_title('Timeline — Anomalies Détectées vs Red Team Réels', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)

# Ajout des séparateurs de jours
for d in range(8):
    ax1.axvline(x=d*24, color='gray', linestyle=':', alpha=0.5)
    if d < 7:
        ax1.text(d*24 + 12, ax1.get_ylim()[1]*0.9, f'Jour {d}', 
                ha='center', fontsize=10, color='gray')

# Score moyen par heure
ax2 = axes[1]
ax2.plot(timeline['time_index'], timeline['mean_score'], 
         color='navy', linewidth=1.5, marker='o', markersize=3)
ax2.fill_between(timeline['time_index'], timeline['mean_score'], 
                 alpha=0.2, color='navy')
ax2.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Seuil (score=0)')
ax2.set_xlabel('Heure (depuis le début)', fontsize=12)
ax2.set_ylabel('Score moyen', fontsize=12)
ax2.set_title('Score Moyen d\'Anomalie par Heure', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)

for d in range(8):
    ax2.axvline(x=d*24, color='gray', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig('../results/figures/anomaly_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure sauvegardée : ../results/figures/anomaly_timeline.png")

In [ ]:
# === 7.2 Heatmap des anomalies par jour et heure ===

heatmap_data = features_df.groupby(['day', 'hour'])['is_anomaly'].mean().reset_index()
heatmap_pivot = heatmap_data.pivot(index='day', columns='hour', values='is_anomaly')

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(heatmap_pivot, cmap='YlOrRd', annot=True, fmt='.2f', 
            linewidths=0.5, ax=ax,
            xticklabels=[f'{h}h' for h in range(24)],
            yticklabels=[f'Jour {d}' for d in sorted(heatmap_data['day'].unique())],
            cbar_kws={'label': 'Taux d\'anomalies'})
ax.set_title('Heatmap — Taux d\'Anomalies par Jour et Heure', fontsize=14, fontweight='bold')
ax.set_xlabel('Heure du jour', fontsize=12)
ax.set_ylabel('', fontsize=12)

plt.tight_layout()
plt.savefig('../results/figures/anomaly_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure sauvegardée : ../results/figures/anomaly_heatmap.png")

In [ ]:
# === 7.3 Scatter plot 2D : failure_ratio vs off_hours_ratio ===

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Par prédiction du modèle
ax1 = axes[0]
normal_mask = features_df['is_anomaly'] == 0
anomaly_mask = features_df['is_anomaly'] == 1

ax1.scatter(features_df.loc[normal_mask, 'failure_ratio'], 
            features_df.loc[normal_mask, 'off_hours_ratio'],
            c='steelblue', alpha=0.3, s=10, label='Normal')
ax1.scatter(features_df.loc[anomaly_mask, 'failure_ratio'], 
            features_df.loc[anomaly_mask, 'off_hours_ratio'],
            c='crimson', alpha=0.6, s=20, label='Anomalie')
ax1.set_xlabel('Failure Ratio', fontsize=12)
ax1.set_ylabel('Off-Hours Ratio', fontsize=12)
ax1.set_title('Prédictions du Modèle', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)

# Par label réel
ax2 = axes[1]
normal_mask_real = features_df['is_red_team'] == 0
rt_mask_real = features_df['is_red_team'] == 1

ax2.scatter(features_df.loc[normal_mask_real, 'failure_ratio'], 
            features_df.loc[normal_mask_real, 'off_hours_ratio'],
            c='steelblue', alpha=0.3, s=10, label='Normal')
ax2.scatter(features_df.loc[rt_mask_real, 'failure_ratio'], 
            features_df.loc[rt_mask_real, 'off_hours_ratio'],
            c='crimson', alpha=0.6, s=20, label='Red Team')
ax2.set_xlabel('Failure Ratio', fontsize=12)
ax2.set_ylabel('Off-Hours Ratio', fontsize=12)
ax2.set_title('Labels Réels (Red Team)', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)

plt.suptitle('Failure Ratio vs Off-Hours Ratio', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/scatter_failure_offhours.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure sauvegardée : ../results/figures/scatter_failure_offhours.png")

In [ ]:
# === 7.4 Importance des features (via permutation) ===

# Calculer l'importance en mesurant l'impact de chaque feature sur le score
feature_importance = []
baseline_scores = iso_forest.decision_function(X_scaled)

for i, feat in enumerate(FEATURE_COLS):
    X_permuted = X_scaled.copy()
    np.random.shuffle(X_permuted[:, i])  # Mélanger la feature
    permuted_scores = iso_forest.decision_function(X_permuted)
    importance = np.mean(np.abs(baseline_scores - permuted_scores))
    feature_importance.append({'feature': feat, 'importance': importance})

importance_df = pd.DataFrame(feature_importance).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(importance_df['feature'], importance_df['importance'], 
        color='teal', alpha=0.8)
ax.set_xlabel('Importance (variation moyenne du score)', fontsize=12)
ax.set_title('Importance des Features — Isolation Forest', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/figures/feature_importance_iforest.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure sauvegardée : ../results/figures/feature_importance_iforest.png")

In [ ]:
# === 7.5 Matrice de corrélation des features ===

fig, ax = plt.subplots(figsize=(10, 8))
corr_matrix = features_df[FEATURE_COLS].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', 
            cmap='RdBu_r', center=0, linewidths=0.5, ax=ax,
            vmin=-1, vmax=1, square=True,
            cbar_kws={'label': 'Corrélation'})
ax.set_title('Matrice de Corrélation des Features', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/figures/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure sauvegardée : ../results/figures/correlation_matrix.png")

## 8. 🔧 Optimisation des Hyperparamètres

Tester différentes valeurs de `contamination` et `n_estimators` pour trouver les meilleurs paramètres.

In [ ]:
# === 8.1 Recherche du meilleur taux de contamination ===

contamination_values = [0.01, 0.02, 0.03, 0.05, 0.07, 0.10, 0.15, 0.20]
results = []

print("🔧 Test de différentes valeurs de contamination...")
print(f"{'Contamination':>15} {'Precision':>10} {'Recall':>10} {'F1-score':>10} {'N_anomalies':>12}")
print("-" * 60)

for cont in contamination_values:
    model = IsolationForest(
        n_estimators=200, contamination=cont,
        random_state=42, n_jobs=-1
    )
    model.fit(X_scaled)
    preds = (model.predict(X_scaled) == -1).astype(int)
    
    p = precision_score(y_true, preds, zero_division=0)
    r = recall_score(y_true, preds, zero_division=0)
    f = f1_score(y_true, preds, zero_division=0)
    n_anom = preds.sum()
    
    results.append({'contamination': cont, 'precision': p, 'recall': r, 
                    'f1_score': f, 'n_anomalies': n_anom})
    print(f"{cont:>15.2f} {p:>10.4f} {r:>10.4f} {f:>10.4f} {n_anom:>12,}")

results_df = pd.DataFrame(results)
best_row = results_df.loc[results_df['f1_score'].idxmax()]
print(f"\n✅ Meilleure contamination : {best_row['contamination']} (F1 = {best_row['f1_score']:.4f})")

In [ ]:
# === 8.2 Visualisation des résultats d'optimisation ===

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(results_df['contamination'], results_df['precision'], 
        'o-', color='blue', linewidth=2, markersize=8, label='Precision')
ax.plot(results_df['contamination'], results_df['recall'], 
        's-', color='green', linewidth=2, markersize=8, label='Recall')
ax.plot(results_df['contamination'], results_df['f1_score'], 
        'D-', color='red', linewidth=2, markersize=8, label='F1-score')

# Marquer le meilleur point
ax.axvline(x=best_row['contamination'], color='gray', linestyle='--', alpha=0.7,
           label=f'Meilleur (c={best_row["contamination"]})')

ax.set_xlabel('Contamination', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Optimisation du Taux de Contamination', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/contamination_optimization.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. 💾 Sauvegarde des Résultats

In [ ]:
# === 9.1 Créer les dossiers de résultats ===
os.makedirs('../results/figures', exist_ok=True)
os.makedirs('../results/metrics', exist_ok=True)

# === 9.2 Sauvegarder les métriques ===
metrics = {
    'model': 'Isolation Forest',
    'n_estimators': 200,
    'contamination': 0.05,
    'precision': precision,
    'recall': recall,
    'f1_score': f1,
    'pr_auc': pr_auc,
    'best_contamination': best_row['contamination'],
    'best_f1_score': best_row['f1_score'],
    'n_total_observations': len(features_df),
    'n_anomalies_detected': features_df['is_anomaly'].sum(),
    'n_red_team_events': features_df['is_red_team'].sum(),
}

metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv('../results/metrics/isolation_forest_metrics.csv', index=False)

# === 9.3 Sauvegarder les prédictions ===
features_df.to_csv('../results/metrics/isolation_forest_predictions.csv', index=False)

# === 9.4 Sauvegarder les résultats d'optimisation ===
results_df.to_csv('../results/metrics/contamination_optimization.csv', index=False)

print("✅ Tous les résultats ont été sauvegardés :")
print("   📄 ../results/metrics/isolation_forest_metrics.csv")
print("   📄 ../results/metrics/isolation_forest_predictions.csv")
print("   📄 ../results/metrics/contamination_optimization.csv")
print("   🖼️  ../results/figures/ (toutes les visualisations)")

## 10. 📝 Résumé et Tableau Récapitulatif

In [ ]:
# === Tableau récapitulatif final ===

print("\n" + "=" * 70)
print("🏆 RÉSUMÉ — MODÈLE ISOLATION FOREST")
print("=" * 70)
print()
print(f"  📊 Dataset         : LANL Cyber-Security Events {'(synthétique)' if not USE_REAL_DATA else '(réel)'}")
print(f"  📈 Observations    : {len(features_df):,} (user × fenêtre temporelle)")
print(f"  🔴 Red Team        : {features_df['is_red_team'].sum():,} ({features_df['is_red_team'].mean():.2%})")
print(f"  🌲 Modèle          : Isolation Forest (200 arbres, contamination=5%)")
print(f"  📏 Features        : {len(FEATURE_COLS)} ({', '.join(FEATURE_COLS)})")
print()
print(f"  📌 PERFORMANCE :")
print(f"     Precision  : {precision:.4f}")
print(f"     Recall     : {recall:.4f}")
print(f"     F1-score   : {f1:.4f}")
print(f"     PR-AUC     : {pr_auc:.4f}")
print()
print(f"  🔧 MEILLEUR PARAMÈTRE :")
print(f"     Contamination optimale : {best_row['contamination']}")
print(f"     F1-score optimal       : {best_row['f1_score']:.4f}")
print()
print("=" * 70)

## 11. 🧠 Conclusion et Interprétation

### Résultats obtenus

L'Isolation Forest a permis de détecter les anomalies dans les logs d'authentification :

1. **Comportements Red Team détectés** : Le modèle identifie les utilisateurs compromis grâce à leurs patterns anormaux :
   - Connexions **hors heures de travail** (nuit, week-end)
   - Accès à **de nombreuses machines inhabituelles**
   - **Taux d'échec élevé** (tentatives de brute force)
   - Utilisation de **types d'authentification variés**

2. **Features les plus discriminantes** : Les features liées au taux d'échec (`failure_ratio`), à l'activité nocturne (`off_hours_ratio`) et au nombre de machines accédées sont les plus importantes.

3. **Choix du seuil de contamination** : Un compromis entre precision et recall est nécessaire selon le contexte opérationnel.

### Avantages de l'Isolation Forest
- ✅ **Non supervisé** : pas besoin de labels pour l'entraînement
- ✅ **Rapide** : complexité linéaire O(n)
- ✅ **Scalable** : fonctionne bien sur de grands volumes de données
- ✅ **Peu de paramètres** : principalement `contamination` et `n_estimators`

### Limites
- ⚠️ Le paramètre `contamination` doit être estimé à l'avance
- ⚠️ Les anomalies subtiles (faibles déviations) peuvent être manquées
- ⚠️ Le modèle ne capture pas les relations temporelles/séquentielles

### Perspectives
- Combiner avec les **Chaînes de Markov** (notebook 06) pour capturer les séquences
- Combiner avec **K-Means** (notebook 07) pour le clustering comportemental
- Tester un **ensemble** combinant les scores des trois modèles

---
*Notebook suivant : `06_markov_chain.ipynb` — Détection d'anomalies par Chaîne de Markov*